# Envio manual de comandos de angulo al ESP32

Envia por Serial el mismo comando que entiende `esp32_servo_control.ino`:

```
<angulo_pin1>,<angulo_pin2>,<angulo_pin3>
```

El ESP32 no se mueve de inmediato: interpola gradualmente desde su posicion
actual hasta la pedida (ver `MOVE_DURATION_MS` en el .ino). Ajusta `PIN_ORDER`
abajo si cambias el orden de pines en el sketch.

In [21]:
import time
import serial
import serial.tools.list_ports
import ipywidgets as widgets
from IPython.display import display

for p in serial.tools.list_ports.comports():
    print(p.device, "-", p.description)

COM5 - Silicon Labs CP210x USB to UART Bridge (COM5)
COM3 - Standard Serial over Bluetooth link (COM3)
COM4 - Standard Serial over Bluetooth link (COM4)


Copia el puerto (ej. `COM5`) de la lista de arriba y ajustalo abajo antes de conectar.

In [22]:
PORT = "COM5"
BAUD = 115200

# Orden de los angulos en el comando, debe coincidir con SERVO_PINS del .ino
PIN_ORDER = [13, 26, 15]

# Los servos de hip y ankle estan montados al reves: la senal que se envia
# por Serial debe ser 180 - angulo_logico. Se aplica en send_angles() y en
# las Opciones C/D via raw_angle_for_pin().
HIP_PIN = 15
ANKLE_PIN = 13
INVERTED_PINS = {HIP_PIN, ANKLE_PIN}

ser = serial.Serial(PORT, BAUD, timeout=2)
time.sleep(2)  # el ESP32 se reinicia al abrir el puerto serie
ser.reset_input_buffer()

ready = ser.readline().decode(errors="replace").strip()
print("Board:", ready if ready else "(sin respuesta, revisa el puerto/baudrate)")

Board: (sin respuesta, revisa el puerto/baudrate)


In [23]:
def raw_angle_for_pin(pin, angle):
    # Hip y ankle estan montados al reves: se manda 180 - angulo logico.
    return 180 - angle if pin in INVERTED_PINS else angle


def send_angles(a1, a2, a3):
    raw = [raw_angle_for_pin(pin, a) for pin, a in zip(PIN_ORDER, (a1, a2, a3))]
    cmd = f"{raw[0]:.2f},{raw[1]:.2f},{raw[2]:.2f}\n"
    ser.reset_input_buffer()
    ser.write(cmd.encode())
    response = ser.readline().decode(errors="replace").strip()
    return cmd.strip(), response


def send_raw(command_text):
    cmd = command_text.strip() + "\n"
    ser.reset_input_buffer()
    ser.write(cmd.encode())
    response = ser.readline().decode(errors="replace").strip()
    return cmd.strip(), response

## Opcion A: escribir el comando a mano (igual que por el Monitor Serial)

Esta opcion manda el texto tal cual por Serial, **sin** pasar por `raw_angle_for_pin`: si escribes un angulo para el pin del hip (15) o del ankle (13), recuerda que esos servos estan invertidos y hay que mandar `180 - angulo_logico` a mano.

In [24]:
command_box = widgets.Text(value="0,0,0", description="Comando:", layout=widgets.Layout(width="300px"))
send_button = widgets.Button(description="Enviar", button_style="primary")
output_box = widgets.Output()


def on_send_clicked(_):
    with output_box:
        sent, response = send_raw(command_box.value)
        print(f"> {sent}")
        print(f"< {response if response else '(sin respuesta)'}")


send_button.on_click(on_send_clicked)
display(widgets.HBox([command_box, send_button]), output_box)

Output()

## Opcion B: sliders por pin (mas comodo para probar angulos)

Los valores de los sliders son angulos logicos; `send_angles` invierte automaticamente los de hip (15) y ankle (13) antes de mandarlos por Serial.

In [4]:
sliders = [
    widgets.IntSlider(value=90, min=0, max=180, description=f"Pin {pin}")
    for pin in PIN_ORDER
]
slider_send_button = widgets.Button(description="Enviar", button_style="primary")
slider_output_box = widgets.Output()


def on_slider_send_clicked(_):
    with slider_output_box:
        sent, response = send_angles(*(s.value for s in sliders))
        print(f"> {sent}")
        print(f"< {response if response else '(sin respuesta)'}")


slider_send_button.on_click(on_slider_send_clicked)
display(widgets.VBox(sliders), slider_send_button, slider_output_box)

Button(button_style='primary', description='Enviar', style=ButtonStyle())

Output()

## Opcion C: reproducir el ciclo (`leg_actuator_angles_one_cycle.csv`, pata FR)

Usa las columnas `fr_hip_joint`, `fr_knee_joint`, `fr_ankle_joint` del CSV
**tal cual, sin restarlas entre si**: `fr_hip_joint` es el angulo absoluto
del primer segmento, y `fr_knee_joint`/`fr_ankle_joint` ya vienen relativos
al segmento anterior (convencion servo hobby, 90° = continuacion recta),
igual que confirma la reconstruccion cinematica en
`plot_leg_actuator_angles.ipynb`. Se mapean a `PIN_ORDER` (hip→15, knee→26,
ankle→13) y se reproducen estirando la duracion original (~0.64 s) a 3 o 5
segundos, para no exigir los servos con un ciclo tan rapido.

Los servos de hip y ankle estan montados al reves: `play_cycle` manda
`raw_angle_for_pin(pin, angulo)` en vez del angulo logico directo, que
invierte (180 - angulo) para los pines 15 y 13.

El `.ino` acepta un 4to campo opcional `<duracion_ms>` en el comando
(`a13,a26,a15,duracion_ms`): es la duracion de la rampa para ESE paso. Sin
ese campo (como en las Opciones A y B) usa el default de 3 s, ideal para
comandos manuales. Aqui cada punto de la trayectoria manda su propia
duracion igual al intervalo entre muestras, para que el ESP32 termine de
interpolar un tramo justo cuando llega el siguiente punto.

In [ ]:
import pandas as pd
import numpy as np

CSV_PATH = "leg_actuator_angles_one_cycle.csv"

df_cycle = pd.read_csv(CSV_PATH)
t_orig = df_cycle["time_s"].values

# Las 3 columnas se usan tal cual: fr_hip_joint es el angulo absoluto del
# primer segmento, y fr_knee_joint / fr_ankle_joint YA vienen relativos al
# segmento anterior (convencion servo hobby, 90 = continuacion recta),
# igual que confirma plot_leg_actuator_angles.ipynb.
PIN_TO_COLUMN = {
    15: "fr_hip_joint",
    26: "fr_knee_joint",
    13: "fr_ankle_joint",
}

for pin, col in PIN_TO_COLUMN.items():
    print(f"pin {pin} ({col}): {df_cycle[col].min():.2f} a {df_cycle[col].max():.2f}")


def build_trajectory(duration_s, n_points=60):
    # Remapea el ciclo original (duracion t_orig[-1]) a duration_s segundos,
    # devolviendo n_points muestras por pin en PIN_ORDER (angulos logicos,
    # sin invertir el hip todavia).
    t_new = np.linspace(0, duration_s, n_points)
    t_lookup = t_new * (t_orig[-1] / duration_s)  # de vuelta al rango temporal original
    values = {
        pin: np.interp(t_lookup, t_orig, df_cycle[PIN_TO_COLUMN[pin]].values)
        for pin in PIN_ORDER
    }
    dt = duration_s / (n_points - 1)
    return values, dt


def play_cycle(duration_s, n_points=60, on_progress=None):
    values, dt = build_trajectory(duration_s, n_points)
    # Duracion de rampa por paso = intervalo entre muestras (en ms), para que
    # el ESP32 termine de interpolar cada tramo justo cuando llega el
    # siguiente punto, en vez de arrastrar la rampa larga de 3 s por defecto.
    step_duration_ms = max(20, round(dt * 1000))

    ser.reset_input_buffer()
    for i in range(n_points):
        angles = ",".join(f"{raw_angle_for_pin(pin, values[pin][i]):.2f}" for pin in PIN_ORDER)
        cmd = f"{angles},{step_duration_ms}"
        ser.write((cmd + "\n").encode())
        if on_progress:
            on_progress(i, n_points, cmd)
        time.sleep(dt)

In [ ]:
duration_dropdown = widgets.Dropdown(
    options=[("3 segundos", 3.0), ("5 segundos", 5.0)],
    value=5.0,
    description="Duracion:",
)
play_button = widgets.Button(description="Reproducir ciclo", button_style="success")
cycle_output = widgets.Output()


def on_play_clicked(_):
    with cycle_output:
        cycle_output.clear_output()
        duration = duration_dropdown.value
        print(f"Reproduciendo one_cycle (pata FR) en {duration:.0f} s...")
        play_cycle(duration)
        print("Listo.")


play_button.on_click(on_play_clicked)
display(widgets.HBox([duration_dropdown, play_button]), cycle_output)

Output()

## Opcion D: reproducir en bucle continuo (con boton Detener)

`leg_actuator_angles_trajectory_adapted.csv` resulta ser exactamente el
`one_cycle` repetido ~95 veces seguidas (mismo angulo cada 0.635 s), asi que
en vez de cargar ese CSV grande, este bucle repite indefinidamente el mismo
ciclo estirado de la Opcion C — el resultado es equivalente, pero mas lento
que el csv (3 o 5 s por vuelta, en vez de 0.635 s) y sin limite de vueltas.

Antes de arrancar el bucle, se manda un comando a la primera posicion de la
trayectoria con una rampa suave de 3 s (la del `.ino` para comandos
manuales) y se espera a que termine; recien ahi arranca el ciclo.

El bucle corre en un hilo aparte para que el boton "Detener" responda de
inmediato aunque la reproduccion este en curso. Evita usar las Opciones A/B
mientras el bucle esta corriendo (competirian por el puerto serie).

In [ ]:
import threading

GO_TO_START_DURATION_MS = 3000


def go_to_start(values):
    angles = ",".join(f"{raw_angle_for_pin(pin, values[pin][0]):.2f}" for pin in PIN_ORDER)
    cmd = f"{angles},{GO_TO_START_DURATION_MS}"
    ser.reset_input_buffer()
    ser.write((cmd + "\n").encode())
    time.sleep(GO_TO_START_DURATION_MS / 1000.0 + 0.1)


def loop_cycle(duration_s, stop_event, n_points=60, on_progress=None):
    values, dt = build_trajectory(duration_s, n_points)
    step_duration_ms = max(20, round(dt * 1000))

    go_to_start(values)

    cycle_num = 0
    while not stop_event.is_set():
        cycle_num += 1
        for i in range(n_points):
            if stop_event.is_set():
                break
            angles = ",".join(f"{raw_angle_for_pin(pin, values[pin][i]):.2f}" for pin in PIN_ORDER)
            cmd = f"{angles},{step_duration_ms}"
            ser.write((cmd + "\n").encode())
            if on_progress:
                on_progress(cycle_num, i, n_points, cmd)
            stop_event.wait(dt)  # duerme el intervalo, pero responde al instante si se pide detener

In [ ]:
loop_duration_dropdown = widgets.Dropdown(
    options=[("3 s por ciclo", 3.0), ("5 s por ciclo", 5.0)],
    value=5.0,
    description="Duracion:",
)
start_loop_button = widgets.Button(description="Iniciar bucle", button_style="success")
stop_loop_button = widgets.Button(description="Detener", button_style="danger", disabled=True)
loop_output = widgets.Output()

loop_thread = None
loop_stop_event = threading.Event()


def on_start_loop_clicked(_):
    global loop_thread, loop_stop_event

    if loop_thread is not None and loop_thread.is_alive():
        return  # ya hay un bucle corriendo

    loop_stop_event = threading.Event()
    duration = loop_duration_dropdown.value

    def progress(cycle_num, i, n_points, cmd):
        with loop_output:
            loop_output.clear_output(wait=True)
            print(f"Ciclo {cycle_num} - punto {i + 1}/{n_points}")
            print(f"> {cmd}")

    def worker():
        with loop_output:
            loop_output.clear_output(wait=True)
            print("Yendo suavemente a la posicion inicial de la trayectoria...")
        try:
            loop_cycle(duration, loop_stop_event, on_progress=progress)
        finally:
            with loop_output:
                print("\nBucle detenido.")
            start_loop_button.disabled = False
            stop_loop_button.disabled = True

    loop_thread = threading.Thread(target=worker, daemon=True)
    start_loop_button.disabled = True
    stop_loop_button.disabled = False
    loop_thread.start()


def on_stop_loop_clicked(_):
    loop_stop_event.set()


start_loop_button.on_click(on_start_loop_clicked)
stop_loop_button.on_click(on_stop_loop_clicked)
display(widgets.HBox([loop_duration_dropdown, start_loop_button, stop_loop_button]), loop_output)

## Volver a posicion de reposo (ankle=0, knee=180, hip=0) y cerrar el puerto

In [ ]:
loop_stop_event.set()
if loop_thread is not None:
    loop_thread.join(timeout=5)  # espera a que el hilo del bucle termine y suelte el puerto

# Angulos logicos de reposo (ankle=0, knee=180, hip=0); raw_angle_for_pin
# invierte hip y ankle (pines 15 y 13) antes de mandarlos por Serial.
rest_angles = {13: 0, 26: 180, 15: 0}
rest_cmd = ",".join(f"{raw_angle_for_pin(pin, rest_angles[pin]):.2f}" for pin in PIN_ORDER)
sent, response = send_raw(rest_cmd)
print(f"> {sent}")
print(f"< {response if response else '(sin respuesta)'}")

time.sleep(3.5)  # espera a que termine el movimiento gradual antes de cerrar
ser.close()